In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets sentencepiece evaluate sacrebleu -q

BASE     = "/content/drive/MyDrive/sign_nlg_project"
PROC_DIR = f"{BASE}/data/processed"
print("Done!")

ValueError: mount failed

In [ ]:
import pandas as pd
import json

# Load custom dataset mới
with open(f"{BASE}/data/custom/custom_dataset_large.json") as f:
    custom = json.load(f)

custom_df = pd.DataFrame(custom)
custom_df["input"]  = "keywords to sentence: " + custom_df["keywords"]
custom_df["target"] = custom_df["sentence"]
custom_df = custom_df[["input", "target"]]

# Nhân 30 lần để model học kỹ
custom_df = pd.concat([custom_df] * 30, ignore_index=True)

# Gộp với ASLG cũ
train_df = pd.read_csv(f"{PROC_DIR}/train.csv")
merged   = pd.concat([train_df, custom_df], ignore_index=True).sample(frac=1, random_state=42)
merged.to_csv(f"{PROC_DIR}/train.csv", index=False)

print(f"ASLG:     {len(train_df):,} mẫu")
print(f"Custom:   {len(custom_df):,} mẫu")
print(f"Total:    {len(merged):,} mẫu")

ASLG:     136,018 mẫu
Custom:   65,850 mẫu
Total:    201,868 mẫu


In [ ]:
from datasets import Dataset, DatasetDict

def load_splits(proc_dir):
    train = pd.read_csv(f"{proc_dir}/train.csv").dropna()
    val   = pd.read_csv(f"{proc_dir}/val.csv").dropna()
    test  = pd.read_csv(f"{proc_dir}/test.csv").dropna()

    ds = DatasetDict({
        "train": Dataset.from_pandas(train[["input","target"]]),
        "val":   Dataset.from_pandas(val[["input","target"]]),
        "test":  Dataset.from_pandas(test[["input","target"]]),
    })
    print(f"Train: {len(ds['train']):,} | Val: {len(ds['val']):,} | Test: {len(ds['test']):,}")
    return ds

ds = load_splits(PROC_DIR)

Train: 201,868 | Val: 8,771 | Test: 8,771


In [ ]:
from transformers import T5Tokenizer

MODEL_OLD = f"{BASE}/models/t5_sign_model"
tokenizer = T5Tokenizer.from_pretrained(MODEL_OLD)

MAX_INPUT  = 64
MAX_TARGET = 64

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input"],
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length",
    )
    label_ids = [
        [-100 if t == tokenizer.pad_token_id else t for t in l]
        for l in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs

tokenized = ds.map(tokenize_batch, batched=True, batch_size=256,
                   remove_columns=["input", "target"])
print("Tokenize xong!")

Map:   0%|          | 0/201868 [00:00<?, ? examples/s]

Map:   0%|          | 0/8771 [00:00<?, ? examples/s]

Map:   0%|          | 0/8771 [00:00<?, ? examples/s]

Tokenize xong!


In [ ]:
from transformers import T5ForConditionalGeneration
import torch

MODEL_OLD = f"{BASE}/models/t5_sign_model"
model     = T5ForConditionalGeneration.from_pretrained(MODEL_OLD)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang dùng: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Load model cũ thành công!")

Loading weights:   0%|          | 0/131 [00:02<?, ?it/s]

Đang dùng: cpu
Load model cũ thành công!


In [ ]:
from transformers import Seq2SeqTrainingArguments

MODEL_NEW = f"{BASE}/models/t5_sign_model_v2"

training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_NEW,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=64,
)
print("Training args OK!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training args OK!


In [ ]:
import evaluate
import numpy as np
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds  = np.clip(preds, 0, tokenizer.vocab_size - 1)
    labels = np.where(labels == -100, tokenizer.pad_token_id, labels)
    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    result = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )
    return {"bleu": round(result["score"], 2)}

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Trainer OK!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Trainer OK!


In [ ]:
print("Bắt đầu train tiếp... (~15 phút)")
trainer.train()

Bắt đầu train tiếp... (~15 phút)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

BASE      = "/content/drive/MyDrive/sign_nlg_project"
MODEL_NEW = f"{BASE}/models/t5_sign_model_v2"

# Load checkpoint epoch 3 (cái đã lưu)
import os
checkpoints = [d for d in os.listdir(MODEL_NEW) if d.startswith("checkpoint")]
checkpoints.sort()
print(f"Checkpoints có: {checkpoints}")
best_ckpt = f"{MODEL_NEW}/{checkpoints[-1]}"
print(f"Load: {best_ckpt}")

tokenizer = T5Tokenizer.from_pretrained(best_ckpt)
model     = T5ForConditionalGeneration.from_pretrained(best_ckpt)
device    = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def predict(keywords):
    input_text = f"keywords to sentence: {keywords}"
    input_ids  = tokenizer(input_text, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        outputs = model.generate(input_ids, max_length=64, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

tests = [
    "i eat kid",
    "i eat toilet",
    "where hospital",
    "i hungry eat",
    "help me please",
    "where bathroom",
    "i deaf sign language",
    "i sick",
    "call ambulance",
    "my head hurt pain",
    "thank you very much",
    "what name",
    "i love school teacher sister sugar daddy",
]

print("\n=== KẾT QUẢ ===")
for kw in tests:
    print(f"  {kw:30} → {predict(kw)}")

Checkpoints có: ['checkpoint-17004', 'checkpoint-8502']
Load: /content/drive/MyDrive/sign_nlg_project/models/t5_sign_model_v2/checkpoint-8502


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


=== KẾT QUẢ ===
  i eat kid                      → I eat my kid.
  i eat toilet                   → I eat in the toilet.
  where hospital                 → Where is the hospital?
  i hungry eat                   → I am hungry and want to eat.
  help me please                 → Please help me.
  where bathroom                 → Where is the bathroom?
  i deaf sign language           → I am deaf and use sign language.
  i sick                         → I am sick.
  call ambulance                 → Please call an ambulance.
  my head hurt pain              → My head hurts and I am in pain.
  thank you very much            → Thank you very much.
  what name                      → What is the name of this?
  i love school teacher sister sugar daddy → I love school teachers my sister sugar daddy.


In [ ]:
trainer.save_model(MODEL_NEW)
tokenizer.save_pretrained(MODEL_NEW)
print(f"Đã lưu model mới tại: {MODEL_NEW}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu model mới tại: /content/drive/MyDrive/sign_nlg_project/models/t5_sign_model_v2


In [ ]:
def predict(keywords):
    input_text = f"keywords to sentence: {keywords}"
    input_ids  = tokenizer(input_text, return_tensors="pt").input_ids.to(device)
    model.to(device)
    with torch.no_grad():
        outputs = model.generate(input_ids, max_length=64, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

tests = [
    "i want toilet",
    "where hospital",
    "i hungry eat",
    "help me please",
    "where bathroom",
    "i deaf sign language",
    "i sick need medicine",
    "call ambulance now",
    "my head hurt pain",
    "thank you very much",
]

print("=== KẾT QUẢ ===")
for kw in tests:
    print(f"  {kw:30} → {predict(kw)}")